# Task 2: Relational Database (MySQL) — Metro Interstate Traffic Volume

This notebook creates the MySQL schema, cleans the raw CSV, loads it into the database, and (4) tests the database with CRUD queries.

In [1]:
import os
from pathlib import Path

import mysql.connector
import pandas as pd
from dotenv import load_dotenv

In [2]:

# Find the project root: walk up until we see the "data" folder
ROOT = Path.cwd()
while not (ROOT / "data").exists():
    ROOT = ROOT.parent

RAW_CSV = ROOT / "data" / "raw" / "Metro_Interstate_Traffic_Volume.csv"
SCHEMA_FILE = ROOT / "database" / "sql" / "schema.sql"

load_dotenv(ROOT / ".env")
conn = mysql.connector.connect(
    host=os.getenv("MYSQL_HOST"),
    port=int(os.getenv("MYSQL_PORT")),
    user=os.getenv("MYSQL_USER"),
    password=os.getenv("MYSQL_PASSWORD"),
)

print("Connected. Project root:", ROOT)

Connected. Project root: /home/jonathan/Documents/studies/School/ML Pipeline/machine_learning_pipeline_formative-one


In [3]:
def run(sql, params=None):
    """Run one SQL statement. SELECTs come back as a DataFrame."""
    cur = conn.cursor()
    cur.execute(sql, params)
    out = None
    if cur.description:
        out = pd.DataFrame(cur.fetchall(), columns=cur.column_names)
    else:
        print(f"OK - {cur.rowcount} row(s) affected")
    conn.commit()
    cur.close()
    return out

## 1. Create the schema

Four tables: `holidays` and `weather_conditions` are lookup tables (each value stored once), `traffic_observations` holds one row per hour, and `weather_readings` holds the weather reports for that hour. An hour can have several weather reports (for example "rain" and "drizzle" at the same time), which is why weather is a separate table. The cell below drops the database first so the notebook can be re-run from the top.

In [4]:
print(SCHEMA_FILE.read_text())   # shows the schema in the notebook output

# Start fresh, then run every statement in schema.sql
run("DROP DATABASE IF EXISTS traffic_db")

sql_text = "\n".join(
    line for line in SCHEMA_FILE.read_text().splitlines()
    if not line.strip().startswith("--")
)
for statement in sql_text.split(";"):
    if statement.strip():
        run(statement)

run("SHOW TABLES")

-- Create the database
CREATE DATABASE IF NOT EXISTS traffic_db;

-- Select the database
USE traffic_db;


-- Holidays table
CREATE TABLE holidays (
    holiday_id INT AUTO_INCREMENT PRIMARY KEY,
    holiday_name VARCHAR(50) NOT NULL UNIQUE
);


-- Weather conditions table
CREATE TABLE weather_conditions (
    condition_id INT AUTO_INCREMENT PRIMARY KEY,
    weather_main VARCHAR(30) NOT NULL,
    weather_description VARCHAR(255) NOT NULL,
    UNIQUE (weather_main, weather_description)
);


-- Traffic observations table
CREATE TABLE traffic_observations (
    obs_id INT AUTO_INCREMENT PRIMARY KEY,
    date_time DATETIME NOT NULL UNIQUE,
    traffic_volume INT NOT NULL,
    holiday_id INT,
    FOREIGN KEY (holiday_id) REFERENCES holidays(holiday_id)
);


-- Weather readings table
CREATE TABLE weather_readings (
    reading_id INT AUTO_INCREMENT PRIMARY KEY,
    obs_id INT NOT NULL,
    condition_id INT NOT NULL,
    temp_k DECIMAL(6,2),
    rain_1h DECIMAL(7,2),
    snow_1h DECIMAL(5,2),

,Tables_in_traffic_db
0,holidays
1,traffic_observations
2,weather_conditions
3,weather_readings


In [5]:
raw = pd.read_csv(RAW_CSV, keep_default_na=False)

print("Rows:", len(raw))
print("Exact duplicate rows:", raw.duplicated().sum())
print("Rows sharing a timestamp:", raw["date_time"].duplicated().sum())
print("temp == 0:", (raw["temp"] == 0).sum())
print("rain_1h > 1000:", (raw["rain_1h"] > 1000).sum())
print("holiday == 'None':", (raw["holiday"] == "None").sum())

low = raw["weather_description"].str.lower()
variants = raw.groupby(low)["weather_description"].unique()
print("\nDescriptions that differ only by capitalization:")
print(variants[variants.apply(len) > 1])

Rows: 48204
Exact duplicate rows: 17
Rows sharing a timestamp: 7629
temp == 0: 10
rain_1h > 1000: 1
holiday == 'None': 48143

Descriptions that differ only by capitalization:
weather_description
sky is clear    [sky is clear, Sky is Clear]
Name: weather_description, dtype: object


In [6]:
def clean(df):
    df = df.copy()
    df["weather_main"] = df["weather_main"].str.strip().str.lower()
    df["weather_description"] = df["weather_description"].str.strip().str.lower()

    df = df.drop_duplicates().copy()
    df["date_time"] = pd.to_datetime(df["date_time"])

    df.loc[df["temp"] == 0, "temp"] = None
    df.loc[df["rain_1h"] > 1000, "rain_1h"] = None

    # mask() is safer than replace(..., None), which can forward-fill in older pandas
    df["holiday"] = df["holiday"].mask(df["holiday"] == "None")
    return df

df = clean(raw)
df = df.astype(object).where(df.notna(), None)   # NaN -> None so MySQL stores NULL
print("Rows after cleaning:", len(df))
df.head()

Rows after cleaning: 48187


,holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,traffic_volume
0,None,288.28,0.0,0.0,40,clouds,scattered clouds,2012-10-02 09:00:00,5545
1,None,289.36,0.0,0.0,75,clouds,broken clouds,2012-10-02 10:00:00,4516
2,None,289.58,0.0,0.0,90,clouds,overcast clouds,2012-10-02 11:00:00,4767
3,None,290.13,0.0,0.0,90,clouds,overcast clouds,2012-10-02 12:00:00,5026
4,None,291.14,0.0,0.0,75,clouds,broken clouds,2012-10-02 13:00:00,4918


## 3. Load the data

Load order matters because of the foreign keys: lookup tables first
(`holidays`, `weather_conditions`), then `traffic_observations`, then `weather_readings`.
Expected row counts: holidays 11, weather_conditions 37, traffic_observations 40,575,
weather_readings 48,187.

In [7]:
cur = conn.cursor()

# Empty the tables so this cell can be re-run safely
cur.execute("SET FOREIGN_KEY_CHECKS = 0")
for t in ["weather_readings", "traffic_observations", "weather_conditions", "holidays"]:
    cur.execute(f"TRUNCATE TABLE {t}")
cur.execute("SET FOREIGN_KEY_CHECKS = 1")
conn.commit()

# 1. holidays
holidays = sorted(h for h in df["holiday"].unique() if h is not None)
cur.executemany("INSERT INTO holidays (holiday_name) VALUES (%s)", [(h,) for h in holidays])

# 2. weather_conditions
conds = df[["weather_main", "weather_description"]].drop_duplicates()
cur.executemany(
    "INSERT INTO weather_conditions (weather_main, weather_description) VALUES (%s, %s)",
    list(conds.itertuples(index=False, name=None)))
conn.commit()

cur.execute("SELECT holiday_name, holiday_id FROM holidays")
holiday_ids = dict(cur.fetchall())
cur.execute("SELECT weather_main, weather_description, condition_id FROM weather_conditions")
cond_ids = {(m, d): i for m, d, i in cur.fetchall()}

# 3. traffic_observations (one row per unique timestamp)
obs = df.drop_duplicates("date_time")
cur.executemany(
    "INSERT INTO traffic_observations (date_time, traffic_volume, holiday_id) VALUES (%s, %s, %s)",
    [(r.date_time.to_pydatetime(), int(r.traffic_volume), holiday_ids.get(r.holiday))
     for r in obs.itertuples()])
conn.commit()

cur.execute("SELECT date_time, obs_id FROM traffic_observations")
obs_ids = dict(cur.fetchall())

# 4. weather_readings (one row per remaining CSV row)
cur.executemany(
    """INSERT INTO weather_readings
       (obs_id, condition_id, temp_k, rain_1h, snow_1h, clouds_all)
       VALUES (%s, %s, %s, %s, %s, %s)""",
    [(obs_ids[r.date_time.to_pydatetime()],
      cond_ids[(r.weather_main, r.weather_description)],
      r.temp, r.rain_1h, r.snow_1h, int(r.clouds_all))
     for r in df.itertuples()])
conn.commit()
cur.close()
print("Load finished")

Load finished


In [8]:
run("""
SELECT 'holidays' AS table_name, COUNT(*) AS row_count FROM holidays
UNION ALL SELECT 'weather_conditions', COUNT(*) FROM weather_conditions
UNION ALL SELECT 'traffic_observations', COUNT(*) FROM traffic_observations
UNION ALL SELECT 'weather_readings', COUNT(*) FROM weather_readings
""")

,table_name,row_count
0,holidays,11
1,weather_conditions,37
2,traffic_observations,40575
3,weather_readings,48187


## 4. Queries: READ

### Query 1: average traffic by weather type
Joins three tables. Note: an hour with two weather reports counts once for each report.

In [9]:
run("""
SELECT wc.weather_main, ROUND(AVG(t.traffic_volume)) AS avg_traffic, COUNT(*) AS n_readings
FROM traffic_observations t
JOIN weather_readings r ON r.obs_id = t.obs_id
JOIN weather_conditions wc ON wc.condition_id = r.condition_id
GROUP BY wc.weather_main
ORDER BY avg_traffic DESC
""")

,weather_main,avg_traffic,n_readings
0,clouds,3618,15158
1,haze,3502,1360
2,rain,3318,5672
3,drizzle,3292,1820
4,smoke,3238,20
5,clear,3056,13384
6,snow,3016,2875
7,thunderstorm,2999,1033
8,mist,2933,5949
9,fog,2704,912


### Query 2: holiday traffic compared with normal days
In this dataset a holiday is only flagged on the **midnight (00:00) row** of that day.
Comparing those rows with all other hours would be unfair, because midnight traffic is
always low. So both sides are limited to `HOUR = 0`.

In [10]:
run("""
SELECT COALESCE(h.holiday_name, 'Not a holiday') AS day_type,
       COUNT(*) AS n_hours,
       ROUND(AVG(t.traffic_volume)) AS avg_midnight_traffic
FROM traffic_observations t
LEFT JOIN holidays h ON h.holiday_id = t.holiday_id
WHERE HOUR(t.date_time) = 0
GROUP BY day_type
ORDER BY avg_midnight_traffic DESC
""")

,day_type,n_hours,avg_midnight_traffic
0,New Years Day,5,1325
1,Memorial Day,5,1108
2,Independence Day,5,1076
3,Labor Day,5,1013
4,Thanksgiving Day,5,954
5,Christmas Day,5,847
6,Not a holiday,1670,844
7,Veterans Day,5,691
8,Washingtons Birthday,5,635
9,State Fair,5,635


### Query 3: the five busiest hours of the day

In [11]:
run("""
SELECT HOUR(date_time) AS hour_of_day, ROUND(AVG(traffic_volume)) AS avg_traffic
FROM traffic_observations
GROUP BY hour_of_day
ORDER BY avg_traffic DESC
LIMIT 5
""")

,hour_of_day,avg_traffic
0,16,5709
1,17,5351
2,15,5272
3,14,4957
4,7,4770


### Query 4: data-quality check
The sensor errors were set to NULL during cleaning. Expected: 10 NULL temperatures,
1 NULL rain value.

In [12]:
run("""
SELECT SUM(temp_k IS NULL) AS null_temp, SUM(rain_1h IS NULL) AS null_rain
FROM weather_readings
""")

,null_temp,null_rain
0,10,1


## 5. CREATE, UPDATE and DELETE

The date 2018-10-01 00:00 is after the last row in the data (2018-09-30 23:00), so we
can safely add and remove it.

### CREATE: add a new observation and its weather reading

In [13]:
run("""INSERT INTO traffic_observations (date_time, traffic_volume, holiday_id)
       VALUES ('2018-10-01 00:00:00', 450, NULL)""")

run("""INSERT INTO weather_readings (obs_id, condition_id, temp_k, rain_1h, snow_1h, clouds_all)
       VALUES (
         (SELECT obs_id FROM traffic_observations WHERE date_time = '2018-10-01 00:00:00'),
         (SELECT condition_id FROM weather_conditions
          WHERE weather_main = 'clear' AND weather_description = 'sky is clear'),
         285.50, 0, 0, 20)""")

run("""SELECT t.obs_id, t.date_time, t.traffic_volume, r.temp_k, wc.weather_description
       FROM traffic_observations t
       JOIN weather_readings r ON r.obs_id = t.obs_id
       JOIN weather_conditions wc ON wc.condition_id = r.condition_id
       WHERE t.date_time = '2018-10-01 00:00:00'""")

OK - 1 row(s) affected
OK - 1 row(s) affected


,obs_id,date_time,traffic_volume,temp_k,weather_description
0,40576,2018-10-01,450,285.50,sky is clear


### UPDATE: correct the traffic volume of that observation

In [14]:
run("UPDATE traffic_observations SET traffic_volume = 475 WHERE date_time = '2018-10-01 00:00:00'")
run("SELECT obs_id, date_time, traffic_volume FROM traffic_observations WHERE date_time = '2018-10-01 00:00:00'")

OK - 1 row(s) affected


,obs_id,date_time,traffic_volume
0,40576,2018-10-01,475


### DELETE: remove the observation
The foreign key from `weather_readings` to `traffic_observations` stops us deleting a parent row that still has children, so the weather reading is deleted first.

In [15]:
run("""DELETE FROM weather_readings
       WHERE obs_id = (SELECT obs_id FROM traffic_observations
                       WHERE date_time = '2018-10-01 00:00:00')""")
run("DELETE FROM traffic_observations WHERE date_time = '2018-10-01 00:00:00'")

# Both counts should be 0 and the totals should be back to 40,575 and 48,187
run("""SELECT
         (SELECT COUNT(*) FROM traffic_observations WHERE date_time = '2018-10-01 00:00:00') AS obs_left,
         (SELECT COUNT(*) FROM traffic_observations) AS total_obs,
         (SELECT COUNT(*) FROM weather_readings) AS total_readings""")

OK - 1 row(s) affected
OK - 1 row(s) affected


,obs_left,total_obs,total_readings
0,0,40575,48187
